# Trích xuất dữ liệu từ file CoNLL (Data Extraction)

In [1]:
import os
import glob
import json

INPUT_DIR = os.path.join("data","VTB-SRL")
OUTPUT_TXT = os.path.join("data","gold","1000_sentences.txt")
OUTPUT_JSON = os.path.join("data","gold","gold_labels.json")
TARGET_SENTENCE_COUNT = 1000

# Cột số 2 (index 1) là cột chứa từ, cột số 13 (index 12) chứa cờ 'Y' (Vị từ)
WORD_COL_INDEX = 1
PREDICATE_COL_INDEX = 12

def process_conll_files():
    sentences_txt = []
    gold_labels_json = []

    # Tìm tất cả các file vtb-*.conll trong thư mục con
    file_pattern = os.path.join(INPUT_DIR, "vtb-*.conll")
    conll_files = glob.glob(file_pattern)

    if not conll_files:
        print(f"Không tìm thấy file nào ở đường dẫn '{file_pattern}'.")
        return

    sentence_count = 0

    for file_path in conll_files:
        if sentence_count >= TARGET_SENTENCE_COUNT:
            break

        with open(file_path, 'r', encoding='utf-8') as f:
            current_sentence_lines = []

            for line in f:
                line = line.strip()

                # Dòng trống đánh dấu kết thúc một câu
                if not line:
                    if current_sentence_lines:
                        has_verb = False
                        words = []
                        tokens_data = []

                        for row in current_sentence_lines:
                            cols = row.split('\t')

                            if len(cols) > WORD_COL_INDEX:
                                word = cols[WORD_COL_INDEX]
                                words.append(word)

                                # Kiểm tra cột 13 (index 12)
                                is_predicate = False
                                if len(cols) > PREDICATE_COL_INDEX and cols[PREDICATE_COL_INDEX] == 'Y':
                                    has_verb = True
                                    is_predicate = True

                                # Lấy các cột nhãn từ cột 14 trở đi
                                srl_labels = cols[PREDICATE_COL_INDEX + 1:] if len(cols) > (PREDICATE_COL_INDEX + 1) else []

                                tokens_data.append({
                                    "word": word,
                                    "is_predicate": is_predicate,
                                    "labels": srl_labels
                                })

                        # Nếu câu thỏa mãn điều kiện, tiến hành lưu trữ
                        if has_verb:
                            full_sentence = " ".join(words)
                            sentences_txt.append(full_sentence)

                            gold_labels_json.append({
                                "id": sentence_count + 1,
                                "sentence": full_sentence,
                                "tokens": tokens_data
                            })

                            sentence_count += 1
                            if sentence_count >= TARGET_SENTENCE_COUNT:
                                break

                    current_sentence_lines = []
                else:
                    current_sentence_lines.append(line)

    # Xuất file Text
    with open(OUTPUT_TXT, 'w', encoding='utf-8') as f_txt:
        for sent in sentences_txt:
            f_txt.write(sent + '\n')

    # Xuất file JSON (Gold Labels)
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f_json:
        json.dump(gold_labels_json, f_json, ensure_ascii=False, indent=4)

    print(f"Đã trích xuất {sentence_count}/{TARGET_SENTENCE_COUNT} câu.")
    print(f"File Text (để dịch): {OUTPUT_TXT}")
    print(f"File JSON (đáp án chuẩn): {OUTPUT_JSON}")

if __name__ == "__main__":
    process_conll_files()

Đã trích xuất 923/1000 câu.
File Text (để dịch): data\gold\1000_sentences.txt
File JSON (đáp án chuẩn): data\gold\gold_labels.json


# Dịch Việt -> Anh (Translation)

In [2]:
'''
Dịch Việt -> Anh sử dụng Google Translate (deep-translator).

Input:
    data/gold/923_sentences.txt

Output:
    data/translation/en_sentences.txt
    data/translation/pairs.json
    data/translation/translation_log.json
'''
import json
import time
import os
from pathlib import Path
from tqdm import tqdm
from deep_translator import GoogleTranslator


INPUT_FILE    = os.path.join("data", "gold", "1000_sentences.txt")
OUTPUT_TXT    = os.path.join("data", "translation", "en_sentences.txt")
OUTPUT_JSON   = os.path.join("data", "translation", "translation_pair.json")
LOG_FILE      = os.path.join("reports", "translation_log.json")

BATCH_SIZE       = 10
DELAY_BATCH      = 1.5
DELAY_RETRY      = 5
MAX_RETRIES      = 3
CHECKPOINT_EVERY = 50


# ĐỌC FILE ĐẦU VÀO
def load_vietnamese_sentences(path: str) -> list[str]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    sentences = [line.strip() for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"Đọc xong {len(sentences)} câu tiếng Việt từ '{path}'")
    return sentences

# LOAD CHECKPOINT (NẾU BỊ GIÁN ĐOẠN)
def load_checkpoint(log_path: str) -> dict:
    p = Path(log_path)
    if p.exists():
        data = json.loads(p.read_text(encoding="utf-8"))
        done = len([x for x in data.get("results", []) if x["en"] is not None])
        print(f"Tìm thấy checkpoint: đã dịch {done} câu — tiếp tục từ đây.")
        return data
    return {"results": [], "errors": []}


def save_checkpoint(log_path: str, state: dict):
    Path(log_path).write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

# DỊCH MỘT CÂU (CÓ THỬ LẠI)
def translate_sentence(translator: GoogleTranslator, vi_text: str, idx: int) -> str | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            en = translator.translate(vi_text)
            if en:
                return en.strip()
        except Exception as e:
            print(f"  Câu {idx} – lần {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(DELAY_RETRY)
    print(f"  Câu {idx}: THẤT BẠI sau {MAX_RETRIES} lần thử.")
    return None

def translate_all(sentences: list[str]) -> list[dict]:
    translator = GoogleTranslator(source="vi", target="en")
    state      = load_checkpoint(LOG_FILE)

    # Xác định điểm bắt đầu từ checkpoint
    done_ids = {r["idx"] for r in state["results"]}
    pending  = [(i, s) for i, s in enumerate(sentences) if i not in done_ids]

    print(f"Cần dịch thêm {len(pending)} câu (bỏ qua {len(done_ids)} câu đã có).")

    with tqdm(total=len(pending), desc="Dịch thuật", unit="câu") as pbar:
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch = pending[batch_start : batch_start + BATCH_SIZE]

            for idx, vi_text in batch:
                en_text = translate_sentence(translator, vi_text, idx)

                record = {"idx": idx, "vi": vi_text, "en": en_text}
                state["results"].append(record)

                if en_text is None:
                    state["errors"].append(idx)

                pbar.update(1)

            # Nghỉ giữa batch
            time.sleep(DELAY_BATCH)

            # Lưu checkpoint định kỳ
            if (batch_start // BATCH_SIZE + 1) % (CHECKPOINT_EVERY // BATCH_SIZE) == 0:
                save_checkpoint(LOG_FILE, state)
                print(f"  Checkpoint lưu tại câu ~{batch_start + BATCH_SIZE}")

    # Lưu checkpoint lần cuối
    save_checkpoint(LOG_FILE, state)

    # Sắp xếp lại theo thứ tự gốc
    state["results"].sort(key=lambda x: x["idx"])
    return state["results"]

# Export output
def save_outputs(results: list[dict]):
    pairs     = []
    en_lines  = []
    failed    = []

    for r in results:
        if r["en"]:
            pairs.append({"vi": r["vi"], "en": r["en"]})
            en_lines.append(r["en"])
        else:
            # Giữ nguyên câu gốc nếu dịch thất bại (để không lệch dòng)
            pairs.append({"vi": r["vi"], "en": "[TRANSLATION_FAILED]"})
            en_lines.append("[TRANSLATION_FAILED]")
            failed.append(r["idx"])

    # File text tiếng Anh (1 câu/dòng)
    Path(OUTPUT_TXT).write_text("\n".join(en_lines), encoding="utf-8")
    print(f"Đã lưu {len(en_lines)} câu tiếng Anh -> '{OUTPUT_TXT}'")

    # File JSON cặp Việt–Anh
    Path(OUTPUT_JSON).write_text(
        json.dumps(pairs, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Đã lưu {len(pairs)} cặp câu -> '{OUTPUT_JSON}'")

    # Báo cáo nhanh
    success_rate = (len(pairs) - len(failed)) / len(pairs) * 100 if pairs else 0
    print("=" * 50)
    print(f"TỔNG KẾT:")
    print(f"  Tổng câu       : {len(pairs)}")
    print(f"  Dịch thành công: {len(pairs) - len(failed)}")
    print(f"  Thất bại       : {len(failed)}")
    print(f"  Tỉ lệ thành công: {success_rate:.1f}%")
    if failed:
        print(f"  Câu thất bại (index): {failed}")
    print("=" * 50)

# ENTRY POINT
def main():
    print("DỊCH VIỆT -> ANH")

    sentences = load_vietnamese_sentences(INPUT_FILE)

    if len(sentences) != 923:
        print(f"Cảnh báo: file chứa {len(sentences)} câu, không phải 923!")

    results = translate_all(sentences)
    save_outputs(results)

    print("HOÀN THÀNH!")
    print(f"File TXT: '{OUTPUT_TXT}'")
    print(f"File JSON: '{OUTPUT_JSON}'")


if __name__ == "__main__":
    main()

DỊCH VIỆT -> ANH
Đọc xong 923 câu tiếng Việt từ 'data\gold\1000_sentences.txt'
Cần dịch thêm 923 câu (bỏ qua 0 câu đã có).


Dịch thuật:   5%|███▋                                                                | 50/923 [00:55<09:51,  1.48câu/s]

  Checkpoint lưu tại câu ~50


Dịch thuật:  11%|███████▎                                                           | 100/923 [01:50<14:13,  1.04s/câu]

  Checkpoint lưu tại câu ~100


...

  Checkpoint lưu tại câu ~850


Dịch thuật:  98%|█████████████████████████████████████████████████████████████████▎ | 900/923 [13:53<00:16,  1.40câu/s]

  Checkpoint lưu tại câu ~900


Dịch thuật: 100%|███████████████████████████████████████████████████████████████████| 923/923 [14:15<00:00,  1.08câu/s]

Đã lưu 923 câu tiếng Anh -> 'data\translation\en_sentences.txt'
Đã lưu 923 cặp câu -> 'data\translation\translation_pair.json'
TỔNG KẾT:
  Tổng câu       : 923
  Dịch thành công: 923
  Thất bại       : 0
  Tỉ lệ thành công: 100.0%
HOÀN THÀNH!
File TXT: 'data\translation\en_sentences.txt'
File JSON: 'data\translation\translation_pair.json'


# Gán nhãn SRL tiếng Anh (English SRL Tagging)

Gán nhãn SRL tiếng anh dùng thư viện allennlp (chỉ dùng được ở phiên bản 3.8) tách riêng ra ở file english_srl.ipynb

# Gióng hàng từ vựng (Word Alignment)

In [ ]:
import json
import os
from simalign import SentenceAligner
from underthesea import word_tokenize

EN_LABELS_FILE = os.path.join("data", "silver", "english_labels_2.json")
VI_SENTENCES_FILE = os.path.join("data", "gold", "1000_sentences.txt")
ALIGNMENT_FILE  = os.path.join("reports", "alignment_2.json")

def load_vi_sentences(filepath):
    """Đọc file txt, mỗi dòng 1 câu tiếng Việt."""
    if not os.path.exists(filepath):
        print(f"ERROR: Không tìm thấy file {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_en_labels(filepath):
    """Đọc english_labels_2.json - danh sách các item {sentence, words, verbs}."""
    if not os.path.exists(filepath):
        print(f"ERROR: Không tìm thấy file {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def tokenize_vietnamese(sentence):
    """Tách từ tiếng Việt bằng underthesea."""
    clean_sentence = sentence.replace("_", " ")
    return word_tokenize(clean_sentence, format="text").split()

def union_alignments(result):
    """Gộp 3 phương pháp alignment (argmax + itermax + match) bằng UNION."""
    seen = set()
    merged = []
    for method in ["argmax", "itermax", "match"]:
        for pair in result.get(method, []):
            key = (pair[0], pair[1])
            if key not in seen:
                seen.add(key)
                merged.append([pair[0], pair[1]])
    merged.sort(key=lambda x: (x[0], x[1]))
    return merged

def main():
    en_data = load_en_labels(EN_LABELS_FILE)
    vi_sentences = load_vi_sentences(VI_SENTENCES_FILE)

    if not en_data or not vi_sentences:
        print("ERROR: Dữ liệu đầu vào bị thiếu. Dừng chương trình.")
        return

    try:
        aligner = SentenceAligner(
            model="bert",
            token_type="bpe",
            matching_methods="aim"
        )
    except Exception as e:
        print(f"ERROR: Khởi tạo SimAlign thất bại: {e}")
        return

    total = min(len(en_data), len(vi_sentences))
    results = []
    skipped = 0

    print(f"Bắt đầu gióng hàng (alignment) cho {total} cặp câu...")

    for i in range(total):
        en_tokens = en_data[i].get("words", [])
        vi_tokens = tokenize_vietnamese(vi_sentences[i])

        if not en_tokens or not vi_tokens:
            skipped += 1
            continue

        try:
            result = aligner.get_word_aligns(en_tokens, vi_tokens)
            align_list = union_alignments(result)
        except Exception as e:
            print(f"WARNING: Lỗi gióng hàng tại câu {i}. Chi tiết: {e}")
            align_list = []
            skipped += 1

        # Tính toán độ bao phủ (coverage)
        vi_cov = (len(set(p[1] for p in align_list)) / len(vi_tokens) * 100) if vi_tokens else 0
        en_cov = (len(set(p[0] for p in align_list)) / len(en_tokens) * 100) if en_tokens else 0

        results.append({
            "sent_id": i,
            "vi_tokens": vi_tokens,
            "en_tokens": en_tokens,
            "pairs": align_list
        })

        # In tiến độ mỗi 50 câu
        if (i + 1) % 50 == 0:
            print(
                f" Đã xử lý [{i + 1}/{total}] | Độ phủ: VI={vi_cov:.0f}%, EN={en_cov:.0f}% | Số cặp từ={len(align_list)}")

    # Tự động tạo thư mục nếu chưa tồn tại
    os.makedirs(os.path.dirname(ALIGNMENT_FILE), exist_ok=True)

    with open(ALIGNMENT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Đã xử lý: {len(results)} câu. Bỏ qua: {skipped} câu.")
    print(f"Kết quả được lưu tại: {ALIGNMENT_FILE}")


if __name__ == "__main__":
    main()

C:\Users\fah4i\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2358.89it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight       

Bắt đầu gióng hàng (alignment) cho 923 cặp câu...
 Đã xử lý [50/923] | Độ phủ: VI=92%, EN=94% | Số cặp từ=17
 Đã xử lý [100/923] | Độ phủ: VI=100%, EN=100% | Số cặp từ=6


# Phóng chiếu nhãn (Label Projection)

In [6]:
import json
import os
from collections import defaultdict, Counter
from simalign import SentenceAligner
from underthesea import word_tokenize

ALIGNMENT_FILE = os.path.join("reports", "alignment_2.json")
EN_LABELS_FILE = os.path.join("data", "silver", "english_labels_2.json")

RAW_OUTPUT = os.path.join("data","silver","silver_raw_2.json")


def normalize_label(bio_label):
    if bio_label == "O":
        return "_"
    raw = bio_label[2:] if bio_label.startswith(("B-", "I-")) else bio_label
    if raw == "V":
        return "V"
    if raw.startswith("ARGM-"):
        return "AM-" + raw[5:]
    if raw.startswith("ARG"):
        return "Arg" + raw[3:]
    return raw


def convert_bio_to_token_dicts(en_item):
    words = en_item.get("words", [])
    verbs = en_item.get("verbs", [])

    if not words or not verbs:
        return []

    num_cols = len(verbs)
    tokens = [{"word": w, "is_predicate": False, "labels": ["_"] * num_cols} for w in words]

    for col_idx, verb_info in enumerate(verbs):
        verb_name = verb_info.get("verb", "unknown")
        tags=verb_info.get("tags", [])

        for tok_idx, tag in enumerate(tags):
            if tok_idx >= len(tokens):
                break
            if tag in ("B-V", "I-V"):
                tokens[tok_idx]["labels"][col_idx] = f"{verb_name}.01"
                tokens[tok_idx]["is_predicate"] = True
            elif tag != "O":
                tokens[tok_idx]["labels"][col_idx] = normalize_label(tag)

    return tokens

def build_vi_to_en_map(pairs):
    vi_to_en = defaultdict(list)
    for pair in pairs:
        en_idx, vi_idx = int(pair[0]), int(pair[1])
        vi_to_en[vi_idx].append(en_idx)
    return vi_to_en

def get_label(vi_idx, col_idx, vi_to_en, en_token_dicts):
    en_indices = vi_to_en.get(vi_idx, [])
    if not en_indices:
        return "_"

    candidates = []
    for en_idx in en_indices:
        if en_idx < len(en_token_dicts):
            labels = en_token_dicts[en_idx].get("labels", [])
            if col_idx < len(labels):
                candidates.append(labels[col_idx])

    if not candidates:
        return "_"
    if len(candidates) == 1:
        return candidates[0]

    # Ưu tiên các nhãn đóng vai trò vị từ (predicate)
    pred = [l for l in candidates if
                        l not in ("_", "") and not l.startswith("Arg") and not l.startswith("AM")]
    if pred:
        return pred[0]

    # Bầu chọn theo số đông (Majority vote)
    non_empty = [l for l in candidates if l != "_"]
    return Counter(non_empty).most_common(1)[0][0] if non_empty else "_"

def main():
    print("Label Projection")

    if not os.path.exists(ALIGNMENT_FILE) or not os.path.exists(EN_LABELS_FILE):
        print("ERROR: Khong tim thay file du lieu dau vao. Vui long kiem tra lai.")
        return

    with open(ALIGNMENT_FILE, "r", encoding="utf-8") as f:
        alignment_data = json.load(f)
    with open(EN_LABELS_FILE, "r", encoding="utf-8") as f:
        en_data = json.load(f)

    silver_raw = []
    skipped = 0

    print(f"Dang xu ly {len(alignment_data)} cau...")

    for item in alignment_data:
        sent_id = item.get("sent_id")
        vi_tokens = item.get("vi_tokens", [])
        pairs = item.get("pairs", [])

        if sent_id >= len(en_data):
            skipped += 1
            continue

        en_token_dicts = convert_bio_to_token_dicts(en_data[sent_id])
        if not en_token_dicts:
            skipped += 1
            continue

        num_cols = len(en_token_dicts[0]["labels"])
        if num_cols == 0:
            skipped += 1
            continue

        vi_to_en = build_vi_to_en_map(pairs)
        result_tokens = []

        for vi_idx, word in enumerate(vi_tokens):
            labels = [get_label(vi_idx, col, vi_to_en, en_token_dicts) for col in range(num_cols)]
            is_predicate = any(l != "_" and ("." in l or l == "V") for l in labels)

            result_tokens.append({
                "word": word,
                "is_predicate": is_predicate,
                "labels": labels
            })

        silver_raw.append({
            "id": sent_id + 1,
            "sentence_vi": " ".join(vi_tokens),
            "tokens": result_tokens
        })

    # Tu dong tao thu muc silver neu chua ton tai
    os.makedirs(os.path.dirname(RAW_OUTPUT), exist_ok=True)

    with open(RAW_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(silver_raw, f, ensure_ascii=False, indent=4)

    print("Phong chieu nhan thanh cong.")
    print(f"Da xu ly: {len(silver_raw)} cau. Bo qua: {skipped} cau.")
    print(f"Ket qua duoc luu tai: {RAW_OUTPUT}")


if __name__ == "__main__":
    main()

Label Projection
Dang xu ly 923 cau...
Phong chieu nhan thanh cong.
Da xu ly: 907 cau. Bo qua: 16 cau.
Ket qua duoc luu tai: data\silver\silver_raw_2.json


# Lọc bằng từ loại (POS Filtering)

In [7]:
"""
Module: filter_pos_tags.py
Description: Filters out erroneously projected predicate labels using Vietnamese POS tagging.
"""

import json
import os
from underthesea import pos_tag

RAW_INPUT = os.path.join("data", "silver", "silver_raw_2.json")

# File đầu ra
CLEANED_OUTPUT = os.path.join("data", "silver", "silver_filtered_2.json")

# Các từ loại (POS tags) không được phép làm vị từ (Predicate) trong tiếng Việt
BLOCKED_POS = {"N", "Np", "M", "P", "E", "C", "T", "L", "CH", "Nc", "Nu"}


def is_predicate_label(label):
    if not label or label == "_":
        return False
    return "." in label or label == "V" or label == "null"

def get_pos_map(vi_tokens):
    """Sử dụng underthesea để gán nhãn từ loại cho câu tiếng Việt."""
    try:
        tagged = pos_tag(" ".join(vi_tokens))
        return {i: tagged[i][1] for i in range(min(len(tagged), len(vi_tokens)))}
    except Exception:
        return {}

def find_predicate_idx(tokens, col_idx):
    """Tìm vị trí của từ đóng vai trò làm vị từ trong một cột nhãn cụ thể."""
    for idx, tok in enumerate(tokens):
        if col_idx < len(tok.get("labels", [])) and is_predicate_label(tok["labels"][col_idx]):
            return idx
    return None

def filter_sentence(item):
    tokens = item.get("tokens", [])
    vi_tokens=[t["word"] for t in tokens]

    if not tokens:
        return None

    num_cols = len(tokens[0]["labels"])
    if num_cols == 0:
        return None

    pos_map = get_pos_map(vi_tokens)
    valid_cols = []

    # Duyệt qua từng cột nhãn (tương ứng với từng vị từ)
    for col_idx in range(num_cols):
        pred_idx = find_predicate_idx(tokens, col_idx)
        if pred_idx is None:
            continue

        pos = pos_map.get(pred_idx, "X")

        # Nếu từ loại KHÔNG nằm trong danh sách cấm -> Cột này hợp lệ
        if pos not in BLOCKED_POS:
            valid_cols.append(col_idx)

    # Nếu câu bị xóa hết mọi cột nhãn (không có vị từ nào hợp lệ) -> Bỏ luôn câu
    if not valid_cols:
        return None

    # Cập nhật lại nhãn cho từng từ, chỉ giữ các cột hợp lệ
    for tok in tokens:
        tok["labels"] = [tok["labels"][i] for i in valid_cols]
        tok["is_predicate"] = any(is_predicate_label(l) for l in tok["labels"])

    return item

def main():
    print("POS Filtering - Underthesea")

    if not os.path.exists(RAW_INPUT):
        print(f"ERROR: Khong tim thay file {RAW_INPUT}")
        print("Goi y: Ban can chay file 'project_srl_labels.py' truoc de sinh ra file nay.")
        return

    with open(RAW_INPUT, "r", encoding="utf-8") as f:
        silver_raw = json.load(f)

    filtered_data = []
    dropped_count = 0
    total_sentences = len(silver_raw)

    print(f"Dang tien hanh loc cho {total_sentences} cau...")

    for i, item in enumerate(silver_raw):
        filtered_item = filter_sentence(item)

        if filtered_item is not None:
            filtered_data.append(filtered_item)
        else:
            dropped_count += 1

        if (i + 1) % 100 == 0:
            print(f" Da xu ly [{i + 1}/{total_sentences}] | Giu lai: {len(filtered_data)} | Loai bo: {dropped_count}")

    # Đánh lại ID cho các câu được giữ lại để đảm bảo tính liên tục
    for new_id, item in enumerate(filtered_data, start=1):
        item["id"] = new_id

    # Tự động tạo thư mục silver nếu chưa tồn tại
    os.makedirs(os.path.dirname(CLEANED_OUTPUT), exist_ok=True)

    with open(CLEANED_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=4)

    print("Loc POS thanh cong.")
    print(f"Tong so cau ban dau : {total_sentences}")
    print(f"So cau giu lai      : {len(filtered_data)} ({(len(filtered_data) / total_sentences) * 100:.1f}%)")
    print(f"So cau bi loai bo   : {dropped_count} ({(dropped_count / total_sentences) * 100:.1f}%)")
    print(f"Ket qua duoc luu tai: {CLEANED_OUTPUT}")


if __name__ == "__main__":
    main()

POS Filtering - Underthesea
Dang tien hanh loc cho 907 cau...
 Da xu ly [100/907] | Giu lai: 92 | Loai bo: 8
 Da xu ly [200/907] | Giu lai: 188 | Loai bo: 12
 Da xu ly [300/907] | Giu lai: 281 | Loai bo: 19
 Da xu ly [400/907] | Giu lai: 375 | Loai bo: 25
 Da xu ly [500/907] | Giu lai: 467 | Loai bo: 33
 Da xu ly [600/907] | Giu lai: 558 | Loai bo: 42
 Da xu ly [700/907] | Giu lai: 645 | Loai bo: 55
 Da xu ly [800/907] | Giu lai: 739 | Loai bo: 61
 Da xu ly [900/907] | Giu lai: 829 | Loai bo: 71
Loc POS thanh cong.
Tong so cau ban dau : 907
So cau giu lai      : 836 (92.2%)
So cau bi loai bo   : 71 (7.8%)
Ket qua duoc luu tai: data\silver\silver_filtered_2.json


In [8]:
import json
import os
from collections import defaultdict

SILVER_FILE = os.path.join("data", "silver", "silver_filtered_2.json")
GOLD_FILE = os.path.join("data", "gold", "gold_labels.json")

REPORT_FILE = os.path.join("reports", "evaluation_report_v2.json")

def flatten(word):
    """Bỏ gạch dưới, chuyển chữ thường để so sánh."""
    return word.replace("_", " ").lower().strip()

def sentence_key(tokens):
    """Tạo key chuẩn hóa từ list tokens để khớp câu."""
    return " ".join(flatten(t["word"]) for t in tokens)

def normalize_label(label):
    """
    Chuẩn hóa nhãn để so sánh công bằng giữa gold (VTB) và silver (PropBank).
    """
    if not label or label == "_":
        return "_"
    lbl = label.strip().lower()

    if lbl.startswith("argm-"):
        return "am-" + lbl[5:]
    if lbl.startswith("arg0-"):
        return "arg0"
    if lbl.startswith("arg1-"):
        return "arg1"
    if lbl.startswith("arg4-"):
        return "arg4"
    if lbl == "arg-location":
        return "arg1"

    return lbl

def align_tokens(gold_tokens, silver_tokens):
    """
    Align token gold <-> silver dựa trên chuỗi ký tự.
    Trả về dict {silver_idx: [gold_idx, ...]}
    """
    silver_to_gold = defaultdict(list)

    # Tạo list từ phẳng của gold
    gold_flat = [flatten(t["word"]) for t in gold_tokens]
    silver_flat = [flatten(t["word"]) for t in silver_tokens]

    g_ptr = 0  # con trỏ gold
    for s_idx, s_word in enumerate(silver_flat):
        s_chars = s_word.replace(" ", "")

        # Gom các gold token cho đến khi khớp với silver token
        matched_chars = ""
        while g_ptr < len(gold_flat) and len(matched_chars) < len(s_chars):
            matched_chars += gold_flat[g_ptr].replace(" ", "")
            silver_to_gold[s_idx].append(g_ptr)
            g_ptr += 1

            if matched_chars == s_chars:
                break

    return silver_to_gold

def build_sentence_map(data, sentence_field):
    smap = {}
    for item in data:
        tokens = item.get("tokens", [])
        key = sentence_key(tokens)
        smap[key] = item
    return smap

def compute_prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return round(p, 4), round(r, 4), round(f1, 4)

def evaluate(silver_data, gold_data):
    gold_map = build_sentence_map(gold_data, "sentence")
    silver_map = build_sentence_map(silver_data, "sentence_vi")

    common_keys = set(gold_map.keys()) & set(silver_map.keys())
    print(f"  Tong so cau trong Gold   : {len(gold_map)}")
    print(f"  Tong so cau trong Silver : {len(silver_map)}")
    print(f"  So cau ghep khop (Match) : {len(common_keys)}\n")

    if not common_keys:
        print("ERROR: Khong ghep khop duoc cau nao giua hai tap du lieu.")
        return None

    pred_tp = pred_fp = pred_fn = 0
    arg_tp = arg_fp = arg_fn = 0
    per_label = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    sentence_stats = []

    for key in sorted(common_keys):
        gold_item = gold_map[key]
        silver_item = silver_map[key]

        gold_tokens = gold_item["tokens"]
        silver_tokens = silver_item["tokens"]

        num_gold_cols = len(gold_tokens[0].get("labels", [])) if gold_tokens else 0
        num_silver_cols = len(silver_tokens[0].get("labels", [])) if silver_tokens else 0

        if num_gold_cols == 0 or num_silver_cols == 0:
            continue

        silver_to_gold = align_tokens(gold_tokens, silver_tokens)

        # 1. Đánh giá phát hiện vị từ (Predicate detection)
        silver_pred_gold_indices = set()
        for s_idx, tok in enumerate(silver_tokens):
            if tok.get("is_predicate"):
                for g_idx in silver_to_gold.get(s_idx, []):
                    silver_pred_gold_indices.add(g_idx)

        gold_pred_indices = {g_idx for g_idx, tok in enumerate(gold_tokens) if tok.get("is_predicate")}

        s_pred_tp = len(silver_pred_gold_indices & gold_pred_indices)
        s_pred_fp = len(silver_pred_gold_indices - gold_pred_indices)
        s_pred_fn = len(gold_pred_indices - silver_pred_gold_indices)

        pred_tp += s_pred_tp
        pred_fp += s_pred_fp
        pred_fn += s_pred_fn

        # 2. Đánh giá gán nhãn tham số (Argument labeling)
        silver_arg_set = set()
        gold_arg_set = set()

        for g_idx, tok in enumerate(gold_tokens):
            for lbl in tok.get("labels", []):
                norm = normalize_label(lbl)
                if norm != "_" and (norm.startswith("arg") or norm.startswith("am-")):
                    gold_arg_set.add((g_idx, norm))

        for s_idx, tok in enumerate(silver_tokens):
            gold_indices = silver_to_gold.get(s_idx, [])
            for s_col, lbl in enumerate(tok.get("labels", [])):
                norm = normalize_label(lbl)
                if norm != "_" and (norm.startswith("arg") or norm.startswith("am-")):
                    for g_idx in gold_indices:
                        silver_arg_set.add((g_idx, norm))

        s_arg_tp = len(silver_arg_set & gold_arg_set)
        s_arg_fp = len(silver_arg_set - gold_arg_set)
        s_arg_fn = len(gold_arg_set - silver_arg_set)

        arg_tp += s_arg_tp
        arg_fp += s_arg_fp
        arg_fn += s_arg_fn

        # Thống kê trên từng nhãn
        all_labels = set(l for _, l in gold_arg_set) | set(l for _, l in silver_arg_set)
        for lbl in all_labels:
            s_set = {i for i, l in silver_arg_set if l == lbl}
            g_set = {i for i, l in gold_arg_set if l == lbl}
            per_label[lbl]["tp"] += len(s_set & g_set)
            per_label[lbl]["fp"] += len(s_set - g_set)
            per_label[lbl]["fn"] += len(g_set - s_set)

        _, _, s_f1 = compute_prf(s_arg_tp, s_arg_fp, s_arg_fn)
        sentence_stats.append({
            "id": gold_item.get("id"),
            "sentence": gold_item.get("sentence", "")[:70],
            "gold_tokens": len(gold_tokens),
            "silver_tokens": len(silver_tokens),
            "pred_tp": s_pred_tp, "pred_fp": s_pred_fp, "pred_fn": s_pred_fn,
            "arg_tp": s_arg_tp, "arg_fp": s_arg_fp, "arg_fn": s_arg_fn,
            "arg_f1": s_f1
        })

    pred_p, pred_r, pred_f1 = compute_prf(pred_tp, pred_fp, pred_fn)
    arg_p, arg_r, arg_f1 = compute_prf(arg_tp, arg_fp, arg_fn)
    overall_f1 = (pred_f1 + arg_f1) / 2

    per_label_results = {}
    for lbl, c in sorted(per_label.items()):
        p, r, f1 = compute_prf(c["tp"], c["fp"], c["fn"])
        per_label_results[lbl] = {
            "precision": p, "recall": r, "f1": f1,
            "tp": c["tp"], "fp": c["fp"], "fn": c["fn"]
        }

    return {
        "summary": {
            "n_gold": len(gold_map),
            "n_silver": len(silver_map),
            "n_matched": len(common_keys),
            "n_evaluated": len(sentence_stats)
        },
        "predicate_detection": {
            "precision": pred_p, "recall": pred_r, "f1": pred_f1,
            "tp": pred_tp, "fp": pred_fp, "fn": pred_fn
        },
        "argument_labeling": {
            "precision": arg_p, "recall": arg_r, "f1": arg_f1,
            "tp": arg_tp, "fp": arg_fp, "fn": arg_fn
        },
        "overall_f1": overall_f1,
        "per_label": per_label_results,
        "sentence_stats": sentence_stats
    }

def print_results(r):
    print("=" * 60)
    print("KET QUA DANH GIA (F1 v2 - Token-Aligned)")
    print("=" * 60)

    s = r["summary"]
    print(f"Gold    : {s['n_gold']} cau")
    print(f"Silver  : {s['n_silver']} cau")
    print(f"Match   : {s['n_matched']} cau")
    print(f"Danh gia: {s['n_evaluated']} cau\n")

    pd = r["predicate_detection"]
    print("PHAT HIEN VI TU (PREDICATE DETECTION)")
    print(f"  P={pd['precision']:.4f}  R={pd['recall']:.4f}  F1={pd['f1']:.4f} "
          f"  (TP={pd['tp']} FP={pd['fp']} FN={pd['fn']})\n")

    al = r["argument_labeling"]
    print("GAN NHAN THAM SO (ARGUMENT LABELING)")
    print(f"  P={al['precision']:.4f}  R={al['recall']:.4f}  F1={al['f1']:.4f} "
          f"  (TP={al['tp']} FP={al['fp']} FN={al['fn']})\n")

    print(f"OVERALL F1 : {r['overall_f1']:.4f}\n")

    print("F1 THEO TUNG LOAI NHAN:")
    print(f"  {'Nhan':<15} {'P':>7} {'R':>7} {'F1':>7} {'TP':>5} {'FP':>5} {'FN':>5}")
    print(f"  {'-' * 57}")
    for lbl, m in sorted(r["per_label"].items(), key=lambda x: -x[1]["f1"]):
        print(f"  {lbl:<15} {m['precision']:>7.4f} {m['recall']:>7.4f} "
              f"{m['f1']:>7.4f} {m['tp']:>5} {m['fp']:>5} {m['fn']:>5}")

    print("\n10 CAU CO F1 THAP NHAT (Can phan tich them):")
    for s in sorted(r["sentence_stats"], key=lambda x: x["arg_f1"])[:10]:
        print(f"  id={str(s['id']):>4} f1={s['arg_f1']:.3f} "
              f"[G={s['gold_tokens']} S={s['silver_tokens']}] {s['sentence'][:55]}")
    print("=" * 60)

def main():
    print("STARTING: Dang doc du lieu...")

    if not os.path.exists(SILVER_FILE):
        print(f"ERROR: Khong tim thay file Silver ({SILVER_FILE})")
        return
    if not os.path.exists(GOLD_FILE):
        print(f"ERROR: Khong tim thay file Gold ({GOLD_FILE})")
        return

    with open(SILVER_FILE, "r", encoding="utf-8") as f:
        silver_data = json.load(f)

    with open(GOLD_FILE, "r", encoding="utf-8") as f:
        gold_data = json.load(f)

    print("Dang tinh toan F1 Score...")
    results = evaluate(silver_data, gold_data)

    if results:
        print_results(results)

        os.makedirs(os.path.dirname(REPORT_FILE), exist_ok=True)
        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        print(f"\nCOMPLETED: Bao cao danh gia da duoc luu tai: {REPORT_FILE}")


if __name__ == "__main__":
    main()

STARTING: Dang doc du lieu...
Dang tinh toan F1 Score...
  Tong so cau trong Gold   : 922
  Tong so cau trong Silver : 835
  So cau ghep khop (Match) : 820

KET QUA DANH GIA (F1 v2 - Token-Aligned)
Gold    : 922 cau
Silver  : 835 cau
Match   : 820 cau
Danh gia: 820 cau

PHAT HIEN VI TU (PREDICATE DETECTION)
  P=0.5680  R=0.5897  F1=0.5786   (TP=1078 FP=820 FN=750)

GAN NHAN THAM SO (ARGUMENT LABELING)
  P=0.5433  R=0.3827  F1=0.4491   (TP=4584 FP=3853 FN=7393)

OVERALL F1 : 0.5139

F1 THEO TUNG LOAI NHAN:
  Nhan                  P       R      F1    TP    FP    FN
  ---------------------------------------------------------
  am-neg           0.7590  0.5164  0.6146    63    20    59
  arg1             0.6447  0.5343  0.5843  2297  1266  2002
  am-tmp           0.6498  0.4431  0.5269   397   214   499
  arg0             0.7661  0.3996  0.5252   917   280  1378
  am-cau           0.7229  0.3659  0.4858    60    23   104
  am-prp           0.6522  0.3779  0.4785   195   104   321
  am-com 